# Coding Exercise: Melbourne Housing Price Prediction

**Dataset:** `./datasets/housing/melb_data.csv` (Melbourne housing)

## Your Task
Build an end-to-end regression workflow to predict house prices using the same ideas from the demo notebook.

1. Load and inspect the data
2. Separate numerical and categorical columns
3. Define features + label (target)
4. Split into train/test
5. Build a preprocessing + model **Pipeline**
6. Evaluate with RMSE
7. Use cross-validation without data leakage
8. Tune with GridSearchCV

## Success Criteria
By the end, you should be able to:
- implement a leakage-safe `Pipeline(preprocess + model)`
- compare model quality using cross-validation RMSE
- report best hyperparameters and final test RMSE

### Rules
- Fill in the code where you see `# TODO:`.
- Do **not** fit preprocessing on the test set.
- For cross-validation and grid search, use a **single Pipeline(preprocess + model)** to avoid leakage.
- Keep `random_state=42` where requested so results are reproducible.

## 1) Imports
Run this cell to import the libraries you will need.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error

## 2) Load the data
Load `melb_data.csv` into a DataFrame named `df`. Then show the first 5 rows and the column names.

In [ ]:
# TODO: load the Melbourne housing dataset
# Path: ./datasets/housing/melb_data.csv
df = ...

# TODO: display basic info
df.head()

### Quick inspection
1. How many rows and columns are there?
2. Which columns have missing values?
3. Which column looks like the target (price)?

In [ ]:
# TODO: inspect the dataset
df.shape

In [ ]:
# TODO: show missing values per column (sorted)
...

## 3) Separate numerical and categorical columns
Before defining features and labels, identify which columns in the dataset are numerical and which are categorical. Use `select_dtypes()` on `df` (excluding the target column `Price`) to separate the columns:
- categorical columns with dtype `object`
- numerical columns with numeric dtypes

This follows the same pattern used in the demo notebook.

In [ ]:
# TODO: identify categorical and numerical columns
categorical_cols = ...
numerical_cols = ...

# Check
assert "Price" in df.columns, "Column 'Price' is missing from df."
categorical_cols = list(categorical_cols)
numerical_cols = list(numerical_cols)
all_cols = set(df.columns)
assert set(categorical_cols).isdisjoint(set(numerical_cols)), "Categorical and numerical columns must not overlap."
assert set(categorical_cols).union(set(numerical_cols)) == all_cols, "Categorical + numerical columns must cover all feature columns (df without Price)."
print("Step 3 check passed.")

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

## 4) Define label (y) and features (X)
We will predict the column `Price`. Create:
- `y` = `df['Price']`
- `X` = all other columns (drop `Price`)

In [ ]:
# TODO: define X and y
y = ...
X = ...

# Check
assert "Price" in df.columns, "Column 'Price' is missing from df."
assert isinstance(X, pd.DataFrame), "X should be a pandas DataFrame."
assert isinstance(y, pd.Series), "y should be a pandas Series."
assert "Price" not in X.columns, "X should not include the target column 'Price'."
assert len(X) == len(y) == len(df), "X, y, and df must have the same number of rows."
print("Step 4 check passed.")

X.head()

## 5) Train/test split
Split into train and test sets:
- 80% train, 20% test
- set `random_state=42`

Create variables: `X_train`, `X_test`, `y_train`, `y_test`.

In [ ]:
# TODO: train/test split
X_train, X_test, y_train, y_test = ...

# Check
assert len(X_train) + len(X_test) == len(X), "Train and test rows should add up to total rows."
assert len(y_train) == len(X_train), "X_train and y_train must have the same number of rows."
assert len(y_test) == len(X_test), "X_test and y_test must have the same number of rows."
assert set(X_train.columns) == set(X.columns), "X_train columns should match original X columns."
assert len(set(X_train.index).intersection(set(X_test.index))) == 0, "Train and test sets should not overlap."
print("Step 5 check passed.")

print(X_train.shape, X_test.shape)

## 6) Build preprocessing (ColumnTransformer)
Use the `categorical_cols` and `numerical_cols` you created earlier.

We will:
- **Numerical columns**: impute missing values (median) + scale (StandardScaler)
- **Categorical columns**: impute missing values (most_frequent) + one-hot encode

### Task
1. Create the numeric preprocessing pipeline.
2. Create the categorical preprocessing pipeline.
3. Build `preprocess = ColumnTransformer(...)` using the earlier column lists.

In [ ]:
# TODO: create numeric and categorical preprocessing pipelines
numeric_transformer = ...
categorical_transformer = ...

# TODO: build the full ColumnTransformer
preprocess = ...

# Check
assert isinstance(preprocess, ColumnTransformer), "preprocess must be a ColumnTransformer."
_ = preprocess.fit_transform(X_train.head(5), y_train.head(5))
print("Step 6b check passed.")

preprocess

## 7) Baseline model (Linear Regression) in a Pipeline
Create a pipeline called `lin_model` that includes:
- `preprocess`
- `LinearRegression()`

Then:
1. Fit on the training set
2. Predict on the test set
3. Compute RMSE on the test set

In [ ]:
# TODO: create + fit the baseline pipeline
lin_model = ...

# TODO: fit on the training set
...

# TODO: predict on the test set
lin_pred = ...

# TODO: compute test RMSE
lin_rmse = ...

# Check
assert isinstance(lin_model, Pipeline), "lin_model must be a sklearn Pipeline."
assert len(lin_pred) == len(y_test), "Number of predictions must match y_test length."
assert np.isfinite(lin_rmse) and lin_rmse > 0, "lin_rmse should be a positive finite number."
print("Step 7 check passed.")
print("Linear Regression Test RMSE:", lin_rmse)

## 8) Cross-validation (no leakage)
Evaluate your **pipeline** using 5-fold cross-validation RMSE.

Reminder: because `lin_model` includes preprocessing + model in one Pipeline, `cross_val_score` will fit preprocessing only on each training fold (no leakage).

In [ ]:
# TODO: cross-validation RMSE (no leakage)
# Hint: use cross_val_score(..., scoring="neg_mean_squared_error", cv=5)
lin_scores = ...
lin_rmse_scores = ...

# Check
assert len(lin_rmse_scores) == 5, "You should have 5 CV RMSE scores for cv=5."
assert np.all(np.isfinite(lin_rmse_scores)), "All CV RMSE scores must be finite."
print("Step 8 check passed.")
print("Linear CV RMSE:", lin_rmse_scores)

## 9) Try two stronger models
Create two more pipelines:
- `tree_model` = DecisionTreeRegressor
- `forest_model` = RandomForestRegressor

### Requirements
- Both must include the same `preprocess` step.
- Set `random_state=42` for reproducibility.
- Evaluate both using 5-fold CV RMSE (same method as before).

In [ ]:
# TODO: Decision Tree pipeline + CV RMSE
# Hint: DecisionTreeRegressor(random_state=42)

tree_model = ...
tree_scores = ...
tree_rmse_scores = ...

# Check
assert len(tree_rmse_scores) == 5, "You should have 5 CV RMSE scores for cv=5."
assert np.all(np.isfinite(tree_rmse_scores)), "All tree CV RMSE scores must be finite."
print("Step 9a check passed.")
print("Tree CV RMSE:", tree_rmse_scores)

In [ ]:
# TODO: Random Forest pipeline + CV RMSE
forest_model = ...
forest_scores = ...
forest_rmse_scores = ...

# Check
assert len(forest_rmse_scores) == 5, "You should have 5 CV RMSE scores for cv=5."
assert np.all(np.isfinite(forest_rmse_scores)), "All forest CV RMSE scores must be finite."
print("Step 9b check passed.")
print("Forest CV RMSE:", forest_rmse_scores)

## 10) Hyperparameter tuning with GridSearchCV
Tune the Random Forest **pipeline** using GridSearchCV.

### Task
1. Import `GridSearchCV`
2. Use a `param_grid` with parameters prefixed by `model__`
   - Example: `model__n_estimators`, `model__max_features`
3. Use `cv=3` to keep runtime reasonable
4. Fit on `X_train`, `y_train`
5. Print the best params and best RMSE (convert from negative MSE)

Tip: start with a small grid first (faster), then expand if needed.

In [ ]:
# TODO: GridSearchCV over the Random Forest pipeline
# Hint: parameters should be prefixed with model__ (e.g., model__n_estimators)

from sklearn.model_selection import GridSearchCV

param_grid = ...

grid_search = ...

grid_search.fit(X_train, y_train)

# Check
if isinstance(param_grid, dict):
    grid_keys = list(param_grid.keys())
else:
    grid_keys = [k for d in param_grid for k in d.keys()]
assert any(k.startswith("model__") for k in grid_keys), "Grid keys must include the 'model__' prefix."
assert hasattr(grid_search, "best_estimator_"), "GridSearchCV did not fit correctly (no best_estimator_ found)."
best_rmse = np.sqrt(-grid_search.best_score_)
assert np.isfinite(best_rmse) and best_rmse > 0, "Best CV RMSE should be a positive finite number."
print("Step 10 check passed.")
print("Best params:", grid_search.best_params_)
print("Best CV RMSE:", best_rmse)

## 11) Final evaluation on the test set
Choose your final model:
- If you did grid search: use `grid_search.best_estimator_`
- Otherwise: use `forest_model` (or your best model)

Then compute RMSE on `X_test` / `y_test`.

In [ ]:
# TODO: pick final model
final_model = ...

# TODO: fit on full training data
final_model.fit(...)

# TODO: evaluate on test set
test_pred = ...
test_rmse = ...

# Check
assert len(test_pred) == len(y_test), "Number of test predictions must match y_test length."
assert np.isfinite(test_rmse) and test_rmse > 0, "test_rmse should be a positive finite number."
print("Step 11 check passed.")
print("Final Test RMSE:", test_rmse)